In [15]:
import pandas as pd
import tqdm
import requests
import json

from utils import MetricHelper, InferenceHelper

import time
import datetime

In [2]:
df = pd.read_csv('Data_FailureFixExplanation.csv')
ref_df = pd.read_csv('Root_Cause_Ref.csv')

In [3]:
# API provided by running llm using Ollama locally
model_deepseek_llm = "deepseek-llm:7b"
model_llama = "llama3.2:3b"
data = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False
}

prompt_task_professional = "###Task:\nYou are a professional software developer. You are given a number of explanations describing the root cause of a software failure. Based on the given explanations, write a single explanation that contains all the information required to understand the root cause of the bug. The explanation should be succinct and without redundant information"

prompt_input = "### Input:\n\nHere are the failure explanations:\n\n"

prompt_output = "### Output:\nFormat your response in valid JSON format with a single field 'explanation' of type string containing your generated explanation."

In [4]:
methods = df['File'].unique().tolist()

ref_c = {method: expl for method, expl in zip(ref_df['bug'].to_list(), ref_df['description_c'].to_list())}
ref_d = {method: expl for method, expl in zip(ref_df['bug'].to_list(), ref_df['description_d'].to_list())}

In [5]:
def generate_explanation(explanations, data, prompt_task, prompt_input='', prompt_output=''):
    url = "http://localhost:11434/api/generate"
    headers = {
        "Content-Type": "application/json"
    }
    input = '\n\n' + '\n\n'.join(["'''\n" + expl + "\n'''" for expl in explanations]) + '\n\n'

    prompt = prompt_task + prompt_input + input + prompt_output
    data['prompt'] = prompt
    response = requests.post(url, headers=headers, data=json.dumps(data))
    return json.loads(response.text)['response']

In [6]:
schema = {
    "$schema": "ase-schema",
    "title": "Explanation",
    "description": "A failure explanation",
    "type": "object",
    "properties": {
        "explanation": {
            "description": "The generated explanation",
            "type": "string"
        }
    },
    "required": ["explanation"]
}
data_json_default = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False,
    "format": "json"
}
data_json = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False,
    "format": schema
}

In [27]:
generated_explanations = {}
for method in tqdm.tqdm(methods):
    method_explanations = df[(df['File'] == method)]['Explanation'].to_list()
    generated_explanations[method] = generate_explanation(method_explanations, data_json, prompt_task_professional, prompt_input, prompt_output)

100%|██████████| 8/8 [09:49<00:00, 73.68s/it] 


In [28]:
generated_explanations.items()

dict_items([('HIT01_8', '{\n    "explanation": "The issue lies on line 279 of the given code, where the variable \\"minutesOffset\\" is incorrectly checked to reject any offset that is less than 0 or greater than 59, while the documentation states that \\"minutesOffset\\" can be negative in some cases. The conditional statement should read \'if (minutesOffset < -59 || minutesOffset > 59).\'"\n}'), ('HIT02_24', '{\n    "explanation": "The failure occurs due to the Color function not accepting negative numbers, which results from passing -0.5 from the getPaint method. Additionally, the variable \'value\' is used instead of \'v\' on line 117, causing an out-of-range error."\n}'), ('HIT03_6', '{\n"explanation": "The StringIndexOutOfBoundsException error occurs due to the codePointAt method throwing out of bounds for index \'pos\', which is incremented too fast, resulting in it being greater than or equal to one for every character in input."\n}'), ('HIT04_7', '{\n"explanation": "There are 

In [ ]:
generated_explanations_dict = {}
for key, value in generated_explanations.items():
    generated_explanations_dict[key] = json.loads(value)['explanation']

In [35]:
generated_explanations_dict

{'HIT01_8': 'The issue lies on line 279 of the given code, where the variable "minutesOffset" is incorrectly checked to reject any offset that is less than 0 or greater than 59, while the documentation states that "minutesOffset" can be negative in some cases. The conditional statement should read \'if (minutesOffset < -59 || minutesOffset > 59).\'',
 'HIT02_24': "The failure occurs due to the Color function not accepting negative numbers, which results from passing -0.5 from the getPaint method. Additionally, the variable 'value' is used instead of 'v' on line 117, causing an out-of-range error.",
 'HIT03_6': "The StringIndexOutOfBoundsException error occurs due to the codePointAt method throwing out of bounds for index 'pos', which is incremented too fast, resulting in it being greater than or equal to one for every character in input.",
 'HIT04_7': 'There are multiple issues affecting the calculation and comparison of maxMiddleIndex within the source code, specifically related to va

In [42]:
generated_explanations_df = pd.DataFrame.from_dict(generated_explanations_dict, orient='index', columns=['explanation'])
generated_explanations_df.to_csv('generated_explanations.csv', header=True)

In [13]:
generated_explanations_df = pd.read_csv('generated_explanations.csv', index_col=0)
generated_explanations_df

,explanation
HIT01_8,"The issue lies on line 279 of the given code, ..."
HIT02_24,The failure occurs due to the Color function n...
HIT03_6,The StringIndexOutOfBoundsException error occu...
HIT04_7,There are multiple issues affecting the calcul...
HIT05_35,The root cause of the software failure is due ...
HIT06_51,The issue arises from the incorrect comparison...
HIT07_33,The bug occurs when the code dereferences memb...
HIT08_54,The failure is caused by the code not being ab...


In [14]:
generated_explanations_combined_df = generated_explanations_df.join(ref_df.set_index('bug')['description_c'])
generated_explanations_combined_df = generated_explanations_combined_df.join(ref_df.set_index('bug')['description_d'])

In [20]:
test = generated_explanations_df.copy()
test = test.join(ref_df.set_index('bug')['description_c']).join(ref_df.set_index('bug')['description_d'])

In [21]:
test

,explanation,description_c,description_d
HIT01_8,"The issue lies on line 279 of the given code, ...",The root cause for the thrown exception is in ...,"The specification states, that the method ""Dat..."
HIT02_24,The failure occurs due to the Color function n...,The root cause for the thrown exception is in ...,"The ""getPaint"" method checks if the input ""val..."
HIT03_6,The StringIndexOutOfBoundsException error occu...,The root cause for the thrown exception is in ...,"The ""translate"" method accesses the character ..."
HIT04_7,There are multiple issues affecting the calcul...,The root cause for the thrown exception is in ...,"The ""updateBounds"" method updates the ""maxMidd..."
HIT05_35,The root cause of the software failure is due ...,The root cause for the thrown error is in line...,"The method ""add"" accepts two input variables ""..."
HIT06_51,The issue arises from the incorrect comparison...,The root cause for the thrown exception is in ...,"The method ""addNumber"" is passed the value ""-0..."
HIT07_33,The bug occurs when the code dereferences memb...,The root cause for the thrown exception is in ...,"The method ""toClass"" accepts the input variabl..."
HIT08_54,The failure is caused by the code not being ab...,The root cause for the thrown error is in line...,"The method """"toLocale"""" accepts a string varia..."


In [15]:
generated_explanations_combined_df

,explanation,description_c,description_d
HIT01_8,"The issue lies on line 279 of the given code, ...",The root cause for the thrown exception is in ...,"The specification states, that the method ""Dat..."
HIT02_24,The failure occurs due to the Color function n...,The root cause for the thrown exception is in ...,"The ""getPaint"" method checks if the input ""val..."
HIT03_6,The StringIndexOutOfBoundsException error occu...,The root cause for the thrown exception is in ...,"The ""translate"" method accesses the character ..."
HIT04_7,There are multiple issues affecting the calcul...,The root cause for the thrown exception is in ...,"The ""updateBounds"" method updates the ""maxMidd..."
HIT05_35,The root cause of the software failure is due ...,The root cause for the thrown error is in line...,"The method ""add"" accepts two input variables ""..."
HIT06_51,The issue arises from the incorrect comparison...,The root cause for the thrown exception is in ...,"The method ""addNumber"" is passed the value ""-0..."
HIT07_33,The bug occurs when the code dereferences memb...,The root cause for the thrown exception is in ...,"The method ""toClass"" accepts the input variabl..."
HIT08_54,The failure is caused by the code not being ab...,The root cause for the thrown error is in line...,"The method """"toLocale"""" accepts a string varia..."


In [11]:
def calculate_metrics(df):
    explanations = df['explanation'].to_list()
    explanations_c = df['description_c'].to_list()
    explanations_d = df['description_d'].to_list()

    df['bleu_c'] = df.apply(lambda row: MetricHelper.calculateBleuScore(row.description_c, row.explanation), axis=1)
    df['bleu_d'] = df.apply(lambda row: MetricHelper.calculateBleuScore(row.description_d, row.explanation), axis=1)
    df['readablity'] = df.apply(lambda row: MetricHelper.calculateReadability(row.explanation), axis=1)
    df['cosine_c'] = df.apply(lambda row: MetricHelper.calculateCosineSimilarity([row.explanation, row.description_c]), axis=1)
    df['cosine_d'] = df.apply(lambda row: MetricHelper.calculateCosineSimilarity([row.explanation, row.description_d]), axis=1)
    df['rouge_c'] = df.apply(lambda row: MetricHelper.calculateRougeScore(row.description_c, row.explanation), axis=1)
    df['rouge_d'] = df.apply(lambda row: MetricHelper.calculateRougeScore(row.description_d, row.explanation), axis=1)
    df['bleurt_c'] = MetricHelper.calculateBleurtScore(explanations_c, explanations)
    df['bleurt_d'] = MetricHelper.calculateBleurtScore(explanations_d, explanations)

    # expl_metrics = {}
    # method_metrics = {}
    # ref_expl_c = ref_c[method]
    # ref_expl_d = ref_d[method]
    # method_metrics['bleu_c'] = MetricHelper.calculateBleuScore(ref_expl_c, expl)
    # method_metrics['bleu_d'] = MetricHelper.calculateBleuScore(ref_expl_d, expl)
    # expl_metrics[method] = method_metrics
    return df

In [12]:
copy_df = calculate_metrics(generated_explanations_combined_df.copy())
copy_df

INFO:tensorflow:Reading checkpoint BLEURT-20.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: BLEURT-20/sent_piece.model.
INFO:tensorflow:SentencePiece tokenizer created.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.
INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:Average batch sequence length: 156.0


INFO:tensorflow:Average batch sequence length: 156.0


INFO:tensorflow:Reading checkpoint BLEURT-20.


...

INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:Average batch sequence length: 250.0


INFO:tensorflow:Average batch sequence length: 250.0


,explanation,description_c,description_d,bleu_c,bleu_d,readablity,cosine_c,cosine_d,rouge_c,rouge_d,bleurt_c,bleurt_d
HIT01_8,"The issue lies on line 279 of the given code, ...",The root cause for the thrown exception is in ...,"The specification states, that the method ""Dat...",0.184985,0.038967,45.09,"[[1.0000000000000002, 0.3602405012135141], [0....","[[1.0000000000000002, 0.3094005542081659], [0....","{'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'r...","{'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'r...",0.541581,0.423919
HIT02_24,The failure occurs due to the Color function n...,The root cause for the thrown exception is in ...,"The ""getPaint"" method checks if the input ""val...",0.018400,0.030063,58.99,"[[0.9999999999999999, 0.224297188640593], [0.2...","[[0.9999999999999999, 0.19957329448517652], [0...","{'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'r...","{'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'r...",0.442805,0.409651
HIT03_6,The StringIndexOutOfBoundsException error occu...,The root cause for the thrown exception is in ...,"The ""translate"" method accesses the character ...",0.005687,0.006991,34.94,"[[1.0, 0.19714265685050741], [0.19714265685050...","[[1.0000000000000002, 0.21252868844149833], [0...","{'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'r...","{'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'r...",0.450107,0.396925
HIT04_7,There are multiple issues affecting the calcul...,The root cause for the thrown exception is in ...,"The ""updateBounds"" method updates the ""maxMidd...",0.012939,0.008639,9.22,"[[1.0000000000000004, 0.0802948533871673], [0....","[[1.0000000000000009, 0.05423515556268173], [0...","{'rouge-1': {'r': 0.13307984790874525, 'p': 0....","{'rouge-1': {'r': 0.125, 'p': 0.00388824884792...",0.398303,0.335807
HIT05_35,The root cause of the software failure is due ...,The root cause for the thrown error is in line...,"The method ""add"" accepts two input variables ""...",0.032762,0.006428,53.89,"[[1.0000000000000002, 0.10808911459059564], [0...","[[0.9999999999999999, 0.023919791509371242], [...","{'rouge-1': {'r': 0.09090909090909091, 'p': 0....","{'rouge-1': {'r': 0.08386075949367089, 'p': 0....",0.454203,0.383946
HIT06_51,The issue arises from the incorrect comparison...,The root cause for the thrown exception is in ...,"The method ""addNumber"" is passed the value ""-0...",0.012232,0.005367,47.42,"[[0.9999999999999996, 0.20947947186868177], [0...","[[0.9999999999999991, 0.14469466010572], [0.14...","{'rouge-1': {'r': 0.08677685950413223, 'p': 0....","{'rouge-1': {'r': 0.0916030534351145, 'p': 0.0...",0.458411,0.354456
HIT07_33,The bug occurs when the code dereferences memb...,The root cause for the thrown exception is in ...,"The method ""toClass"" accepts the input variabl...",0.044884,0.089666,59.98,"[[1.0, 0.36133466577609646], [0.36133466577609...","[[1.0000000000000002, 0.37410440556532193], [0...","{'rouge-1': {'r': 0.06936416184971098, 'p': 0....","{'rouge-1': {'r': 0.07604562737642585, 'p': 0....",0.653411,0.597009
HIT08_54,The failure is caused by the code not being ab...,The root cause for the thrown error is in line...,"The method """"toLocale"""" accepts a string varia...",0.043147,0.023813,67.93,"[[1.0000000000000002, 0.2048514997205282], [0....","[[0.9999999999999993, 0.18291787524207379], [0...","{'rouge-1': {'r': 0.07142857142857142, 'p': 0....","{'rouge-1': {'r': 0.08060453400503778, 'p': 0....",0.383089,0.388746


In [5]:
new_df = InferenceHelper.generateExplanationsWithMetrics(methods, df, ref_df, prompt_task_professional, prompt_input, prompt_output)

INFO:tensorflow:Reading checkpoint BLEURT-20.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: BLEURT-20/sent_piece.model.
INFO:tensorflow:SentencePiece tokenizer created.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.
INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:Average batch sequence length: 223.0


INFO:tensorflow:Average batch sequence length: 223.0


INFO:tensorflow:Reading checkpoint BLEURT-20.


...

INFO:tensorflow:Loading model.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:Average batch sequence length: 278.0


INFO:tensorflow:Average batch sequence length: 278.0


In [21]:
datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

'2025-02-27_16-15-40'

In [24]:
new_df.to_csv('output/' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + '.csv')